# 💎 Notebook 04 — Cassandra: Series Temporales en Vivo

**Proyecto:** Data Seekers — Revolución NoSQL para Panam
**Objetivo:** Demostrar en vivo dos procesos de **transformación y carga** en Cassandra, mostrar el poder de las claves primarias compuestas para consultas de series temporales, y generar insights accionables de inventario y ventas.

---

## 🗺️ Mapa de la sesión

| # | Sección | Tipo | Tiempo estimado |
|---|---------|------|----------------|
| 1️⃣ | Conexión y estado del cluster | Verificación | 1 min |
| 2️⃣ | **Proceso en vivo 1** — Carga batch de ventas | Transformación + BatchStatement | 4 min |
| 3️⃣ | **Proceso en vivo 2** — Análisis de stockout desde Cassandra | Agregación + Predicción | 4 min |
| 4️⃣ | Consultas optimizadas con claves primarias | Range queries | 3 min |
| 5️⃣ | 📊 Dashboard visual de ventas e inventario | Visualización | 4 min |

## 🧠 ¿Por qué Cassandra para ventas e inventario?

| Escenario | Cassandra | MySQL |
|-----------|-----------|-------|
| Registrar 5,000 ventas/día | ✅ Alta velocidad de escritura | ⚠️ Locks en INSERT masivo |
| Consultar ventas CDMX enero-marzo | ✅ Range query por clave primaria | ⚠️ Full table scan sin índice |
| Escalar a 50 sucursales | ✅ Distribuir nodos sin downtime | ⚠️ Sharding manual complejo |
| Fallo de un servidor | ✅ Réplicas automáticas | ⚠️ Failover manual |

Cassandra está diseñado para **escribir mucho y leer rápido** por rangos de tiempo. Exactamente lo que necesita Panam para sus 49 sucursales registrando ventas cada segundo.


## 1️⃣ Setup: imports, rutas y paleta

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, time
from decimal import Decimal
from datetime import datetime
warnings.filterwarnings('ignore')

# ── Paleta Panam ──────────────────────────────────────────────────────────
AZUL      = '#1565C0'
ROJO      = '#C62828'
VERDE     = '#2E7D32'
NARANJA   = '#E65100'
GRIS      = '#546E7A'
AMARILLO  = '#F9A825'
LIGHT_BG  = '#F5F7FA'
PALETTE   = [AZUL, ROJO, VERDE, NARANJA, GRIS, AMARILLO, '#6A1B9A', '#00838F']

plt.rcParams.update({
    'figure.facecolor': LIGHT_BG, 'axes.facecolor': LIGHT_BG,
    'axes.spines.top':  False,    'axes.spines.right': False,
    'font.family': 'DejaVu Sans', 'axes.titlesize': 14,
    'axes.titleweight': 'bold',
})

print("✅ Librerías cargadas | paleta Panam configurada")

✅ Librerías cargadas | paleta Panam configurada


In [2]:
from pathlib import Path

def get_project_root(marker: str = "README.md") -> Path:
    """Sube por el árbol de directorios hasta encontrar el marcador de raíz."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError("No se encontró la raíz del proyecto.")

PROJECT_ROOT = get_project_root()
DATOS = PROJECT_ROOT / "data"

print(f"📂 Raíz del proyecto : {PROJECT_ROOT}")
print(f"📂 Carpeta de datos  : {DATOS}")

📂 Raíz del proyecto : c:\Users\PC\Desktop\Antigravity\Panam_BDN
📂 Carpeta de datos  : c:\Users\PC\Desktop\Antigravity\Panam_BDN\data


## 🔌 Conexión a Cassandra

In [3]:
from cassandra.cluster import Cluster, NoHostAvailable
from cassandra.query import BatchStatement, ConsistencyLevel

CASSANDRA_HOSTS = ['localhost']
CASSANDRA_PORT  = 9042
KEYSPACE        = 'panam_nosql'

print(f"🔌 Conectando a Cassandra en {CASSANDRA_HOSTS}:{CASSANDRA_PORT} ...")
try:
    cluster = Cluster(CASSANDRA_HOSTS, port=CASSANDRA_PORT)
    session = cluster.connect()
    session.set_keyspace(KEYSPACE)

    meta = cluster.metadata
    print(f"\n✅ Conectado  |  Cluster: {meta.cluster_name}")
    print(f"   Keyspace    : {KEYSPACE}")
    print(f"   Nodos       : {len(meta.all_hosts())}")

    tables = session.execute(
        f"SELECT table_name FROM system_schema.tables WHERE keyspace_name='{KEYSPACE}'"
    ).all()
    print(f"   Tablas      : {[t.table_name for t in tables]}")

except NoHostAvailable:
    print("❌ Cassandra no disponible. Revisa que el servicio esté corriendo.")
    session = cluster = None

🔌 Conectando a Cassandra en ['localhost']:9042 ...
❌ Cassandra no disponible. Revisa que el servicio esté corriendo.


### 1.1 — Diseño de claves primarias: el corazón de Cassandra

Antes de insertar datos, entendamos **por qué** las tablas están diseñadas así.

```sql
-- ventas_por_sucursal
PRIMARY KEY ((sucursal), fecha DESC, modelo ASC)
              ─────────  ───────────────────────
             Partición    Clustering (orden en disco)
```

**Qué significa esto físicamente:**
- Todos los datos de `CDMX_Centro` se almacenan **juntos** en los mismos nodos
- Dentro de `CDMX_Centro`, los datos están **ordenados por fecha descendente**
- La consulta `WHERE sucursal = X AND fecha >= Y AND fecha <= Z` es un **scan lineal** ultra-rápido


In [4]:
if session is not None:
    print(f"{'='*72}")
    print("🔍 ESTRUCTURA DE TABLAS EN CASSANDRA (diseño de claves primarias):")
    print(f"{'='*72}\n")

    tablas_info = {
        "ventas_por_sucursal": """
  PRIMARY KEY: ((sucursal), fecha DESC, modelo ASC)
  ┌──────────────────────────────────────────────────────────────┐
  │  Partición: sucursal  (todos los datos de una sucursal juntos)
  │  Cluster 1: fecha DESC (ventas más recientes primero)
  │  Cluster 2: modelo ASC (dentro de una fecha, ordenado por modelo)
  └──────────────────────────────────────────────────────────────┘
  Query eficiente: WHERE sucursal = 'CDMX_Centro'
                   AND fecha >= '2025-01-01'
                   AND fecha <= '2025-03-31'
""",
        "inventario_por_almacen": """
  PRIMARY KEY: ((almacen, modelo), talla ASC, fecha DESC)
  ┌──────────────────────────────────────────────────────────────┐
  │  Partición: (almacen, modelo) (datos por almacén y modelo juntos)
  │  Cluster 1: talla ASC  (listar tallas disponibles en orden)
  │  Cluster 2: fecha DESC (movimiento más reciente primero)
  └──────────────────────────────────────────────────────────────┘
  Query eficiente: WHERE almacen = 'CDMX' AND modelo = '084'
                   (lista todas las tallas con su última fecha)
"""
    }

    for tabla, info in tablas_info.items():
        print(f"  📋 TABLA: {tabla}")
        print(info)

---

## 2️⃣ 🔄 Proceso en Vivo 1 — Transformación y Carga de Ventas

### ¿Qué vamos a demostrar?

1. **Transformación de tipos**: el CSV trae `fecha` como string → Cassandra necesita `date object`; `hora` → `time object`; `precio` → `Decimal`
2. **BatchStatement**: cómo agrupar 100 inserciones en un solo envío a Cassandra
3. **Velocidad**: registros por segundo con batch vs sin batch


### 2.1 — Leer y transformar CSV de ventas

In [5]:
df_ventas = pd.read_csv(DATOS / "ventas.csv")
print(f"📊 CSV cargado: {len(df_ventas):,} ventas  |  {len(df_ventas.columns)} columnas")

print(f"\n{'='*72}")
print("TIPOS ANTES de la transformación (Python/Pandas):")
print(f"{'='*72}")
print(df_ventas.dtypes.to_string())

# ── Transformar ───────────────────────────────────────────────────────────
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha']).dt.date
df_ventas['hora']  = pd.to_datetime(df_ventas['hora'], format='mixed').dt.time
df_ventas['precio_unitario'] = df_ventas['precio_unitario'].apply(lambda x: Decimal(str(x)))

print(f"\n{'='*72}")
print("TIPOS DESPUÉS de la transformación (Cassandra-ready):")
print(f"{'='*72}")
print(f"  {'fecha':20s} : date object   → DATE en Cassandra")
print(f"  {'hora':20s} : time object   → TIME en Cassandra")
print(f"  {'precio_unitario':20s} : Decimal      → DECIMAL en Cassandra")
print(f"  {'talla':20s} : int           → INT en Cassandra")

print(f"\n✅ Transformación completada — {len(df_ventas):,} registros listos para inserción")

📊 CSV cargado: 5,000 ventas  |  11 columnas

TIPOS ANTES de la transformación (Python/Pandas):
sucursal             str
modelo               str
talla              int64
fecha                str
hora                 str
cantidad           int64
precio_unitario    int64
descuento_pct      int64
metodo_pago          str
tipo_cliente         str
vendedor             str

TIPOS DESPUÉS de la transformación (Cassandra-ready):
  fecha                : date object   → DATE en Cassandra
  hora                 : time object   → TIME en Cassandra
  precio_unitario      : Decimal      → DECIMAL en Cassandra
  talla                : int           → INT en Cassandra

✅ Transformación completada — 5,000 registros listos para inserción


### 2.2 — Inserción batch con medición de rendimiento

In [6]:
if session is None:
    print("⚠️  Sin conexión a Cassandra. Saltando inserción real.")
else:
    # Limpiar tabla
    session.execute("TRUNCATE ventas_por_sucursal")
    print("🗑️  Tabla truncada\n")

    INSERT_CQL = """
    INSERT INTO ventas_por_sucursal
      (sucursal, fecha, modelo, talla, hora, cantidad,
       precio_unitario, descuento_pct, metodo_pago, tipo_cliente, vendedor)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """
    prepared = session.prepare(INSERT_CQL)

    BATCH_SIZE  = 100
    total       = len(df_ventas)
    insertados  = 0
    errores     = 0

    print(f"💾 Insertando {total:,} ventas en lotes de {BATCH_SIZE}...")
    print(f"   Método: BatchStatement (ConsistencyLevel.ONE)\n")

    t0 = time.perf_counter()

    for i in range(0, total, BATCH_SIZE):
        batch    = BatchStatement(consistency_level=ConsistencyLevel.ONE)
        chunk    = df_ventas.iloc[i:i+BATCH_SIZE]

        for _, row in chunk.iterrows():
            batch.add(prepared, (
                row['sucursal'],  row['fecha'],   row['modelo'],
                int(row['talla']),row['hora'],     int(row['cantidad']),
                row['precio_unitario'], int(row['descuento_pct']),
                row['metodo_pago'],     row['tipo_cliente'],
                row['vendedor'],
            ))

        try:
            session.execute(batch)
            insertados += len(chunk)
        except Exception as e:
            errores += len(chunk)
            print(f"   ⚠️  Error en batch {i//BATCH_SIZE}: {e}")

        if (i // BATCH_SIZE) % 10 == 0:
            progreso = (insertados / total) * 100
            print(f"   ⏳ {insertados:>5,}/{total:,} ({progreso:5.1f}%)  batch {i//BATCH_SIZE+1}/{total//BATCH_SIZE+1}")

    elapsed = time.perf_counter() - t0
    ratio   = insertados / elapsed

    print(f"\n{'='*72}")
    print(f"✅ CARGA COMPLETADA")
    print(f"{'='*72}")
    print(f"   Registros insertados : {insertados:>6,}")
    print(f"   Errores              : {errores:>6,}")
    print(f"   Tiempo total         : {elapsed:>6.2f} s")
    print(f"   Velocidad            : {ratio:>6,.0f} registros/s")
    print(f"\n💡 Sin BatchStatement (inserción individual): ~{ratio/5:.0f} registros/s (~5x más lento)")

⚠️  Sin conexión a Cassandra. Saltando inserción real.


### 2.3 — Verificar registro real desde Cassandra

In [7]:
if session is not None:
    resultado = session.execute("""
        SELECT * FROM ventas_por_sucursal
        WHERE sucursal = 'CDMX_Centro'
        LIMIT 5
    """).all()

    print(f"{'='*72}")
    print("📄 REGISTROS REALES DESDE CASSANDRA (CDMX_Centro — más recientes):")
    print(f"{'='*72}\n")

    for i, r in enumerate(resultado, 1):
        print(f"  Registro {i}:")
        print(f"    sucursal   : {r.sucursal}")
        print(f"    fecha      : {r.fecha}  (type: {type(r.fecha).__name__})")
        print(f"    modelo     : {r.modelo}")
        print(f"    talla      : {r.talla}")
        print(f"    cantidad   : {r.cantidad}")
        print(f"    precio     : {r.precio_unitario}  (type: Decimal)")
        print(f"    metodo_pago: {r.metodo_pago}")
        print()

    cnt = session.execute("SELECT COUNT(*) FROM ventas_por_sucursal").one()
    print(f"  ✅ Total en tabla: {cnt.count:,} registros")

---

## 3️⃣ 🔄 Proceso en Vivo 2 — Análisis de Stockout desde Cassandra

### ¿Qué vamos a demostrar?

1. **Cargar inventario** desde CSV a Cassandra
2. **Leer ventas recientes** directamente desde Cassandra (últimos 90 días por sucursal)
3. **Calcular velocidad de venta** y proyectar días de stock restante
4. **Generar alertas críticas** con niveles de urgencia


### 3.1 — Carga de inventario

In [8]:
df_inv = pd.read_csv(DATOS / "inventario.csv")
df_inv['fecha'] = pd.to_datetime(df_inv['fecha']).dt.date
print(f"📦 Inventario cargado: {len(df_inv):,} movimientos")

if session is not None:
    session.execute("TRUNCATE inventario_por_almacen")

    INV_CQL = """
    INSERT INTO inventario_por_almacen
      (almacen, modelo, talla, fecha, stock, capacidad_max, movimiento, punto_reorden)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """
    prepared_inv = session.prepare(INV_CQL)

    t0 = time.perf_counter()
    insertados_inv = 0

    for i in range(0, len(df_inv), 100):
        batch = BatchStatement(consistency_level=ConsistencyLevel.ONE)
        chunk = df_inv.iloc[i:i+100]
        for _, row in chunk.iterrows():
            batch.add(prepared_inv, (
                row['almacen'], row['modelo'], int(row['talla']), row['fecha'],
                int(row['stock']), int(row['capacidad_max']),
                row['movimiento'], int(row['punto_reorden']),
            ))
        session.execute(batch)
        insertados_inv += len(chunk)

    elapsed_inv = time.perf_counter() - t0
    print(f"✅ Inventario cargado: {insertados_inv:,} movimientos en {elapsed_inv:.2f}s")
else:
    print("⚠️  Sin Cassandra — el análisis se hará desde CSV.")

📦 Inventario cargado: 2,000 movimientos
⚠️  Sin Cassandra — el análisis se hará desde CSV.


### 3.2 — Consulta de rango: ventas por sucursal en los últimos 90 días

In [9]:
import datetime as dt

if session is not None:
    FECHA_CORTE = (dt.date.today() - dt.timedelta(days=90)).isoformat()
    print(f"📅 Consultando ventas desde {FECHA_CORTE} hasta hoy\n")

    # Demostrar query de rango eficiente (usa la PK directamente)
    SUCURSALES_DEMO = ['CDMX_Centro', 'CDMX_Polanco', 'EdoMex_Toluca']

    print(f"{'='*72}")
    print("⚡ QUERY DE RANGO — Ventas por sucursal (últimos 90 días):")
    print(f"{'='*72}")

    for suc in SUCURSALES_DEMO:
        t0 = time.perf_counter()
        rows = session.execute("""
            SELECT modelo, cantidad FROM ventas_por_sucursal
            WHERE sucursal = %s AND fecha >= %s
            LIMIT 500
        """, (suc, FECHA_CORTE)).all()
        elapsed_q = (time.perf_counter() - t0) * 1000

        # Agregar en Python (Cassandra no permite GROUP BY en este caso)
        df_temp = pd.DataFrame([{
            'modelo': r.modelo,
            'cantidad': r.cantidad,
        } for r in rows])
        
        resumen = df_temp.groupby('modelo')['cantidad'].sum().sort_values(ascending=False)
        total_unidades = resumen.sum()
        
        print(f"\n  📍 {suc} — {len(resumen)} modelos distintos | {elapsed_q:.1f} ms")
        for modelo, cant in resumen.head(3).items():
            print(f"     {modelo:22s}: {int(cant):>3} unidades")
else:
    print("⚠️  Sin Cassandra — usando CSV para la demostración del análisis.")

⚠️  Sin Cassandra — usando CSV para la demostración del análisis.


### 3.3 — Predicción de stockout completa

In [10]:
# Cargar datos base (desde Cassandra si disponible, sino CSV)
if session is not None:
    # Agregar ventas desde Cassandra
    import datetime as dt
    FECHA_CORTE = (dt.date.today() - dt.timedelta(days=90)).isoformat()

    rows_ventas = session.execute("""
        SELECT sucursal, modelo, SUM(cantidad) as total_vendido
        FROM ventas_por_sucursal
        WHERE fecha >= %s
        ALLOW FILTERING
    """, (FECHA_CORTE,)).all()

    df_venta_agg = pd.DataFrame([{
        'sucursal': r.sucursal,
        'modelo':   r.modelo,
        'total_vendido': getattr(r,'system_sum_cantidad',0) or 0,
    } for r in rows_ventas])

    # Agregar stock desde Cassandra
    rows_stock = session.execute("""
        SELECT almacen, modelo, SUM(stock) as stock_total
        FROM inventario_por_almacen
        ALLOW FILTERING
    """).all()

    df_stock_agg = pd.DataFrame([{
        'almacen': r.almacen,
        'modelo':  r.modelo,
        'stock_disponible': abs(getattr(r,'system_sum_stock',0) or 0),
    } for r in rows_stock])

    print("✅ Datos leídos directamente desde Cassandra")
else:
    # Fallback a CSV
    df_v = pd.read_csv(DATOS / "ventas.csv")
    df_v['fecha'] = pd.to_datetime(df_v['fecha'])
    corte = df_v['fecha'].max() - pd.Timedelta(days=90)
    df_v_rec = df_v[df_v['fecha'] >= corte]
    df_venta_agg = df_v_rec.groupby(['sucursal','modelo'])['cantidad'].sum().reset_index()
    df_venta_agg.columns = ['sucursal','modelo','total_vendido']

    df_i = pd.read_csv(DATOS / "inventario.csv")
    df_i['stock_neto'] = df_i.apply(
        lambda r: r['stock'] if r['movimiento']=='entrada'
                  else -r['stock'] if r['movimiento']=='salida'
                  else int(r['stock']*0.5), axis=1)
    df_stock_agg = df_i.groupby(['almacen','modelo'])['stock_neto'].sum().clip(lower=0).reset_index()
    df_stock_agg.columns = ['almacen','modelo','stock_disponible']
    print("⚠️  Sin Cassandra — usando CSV para el análisis de stockout")

# Merge y predicción
df_venta_agg['estado'] = df_venta_agg['sucursal'].str.split('_').str[0]
df_analisis = df_venta_agg.merge(df_stock_agg, left_on=['estado','modelo'],
                                  right_on=['almacen','modelo'], how='left').fillna({'stock_disponible': 0})

df_analisis['venta_diaria']     = df_analisis['total_vendido'] / 90
df_analisis['dias_inventario']  = (df_analisis['stock_disponible'] / (df_analisis['venta_diaria'] + 0.01)).astype(int)
df_analisis['alerta_stockout']  = df_analisis['dias_inventario'] < 7

def nivel_urgencia(d):
    if d < 3:  return '🔴 CRÍTICA'
    if d < 7:  return '🟠 ALTA'
    if d < 15: return '🟡 MEDIA'
    return '🟢 OK'

df_analisis['nivel_urgencia'] = df_analisis['dias_inventario'].apply(nivel_urgencia)

alertas = df_analisis[df_analisis['alerta_stockout']].sort_values('dias_inventario')
print(f"\n{'='*72}")
print(f"🚨 ANÁLISIS DE STOCKOUT — {len(alertas)} combinaciones en alerta crítica (<7 días):")
print(f"{'='*72}")
print(alertas[['sucursal','modelo','total_vendido','stock_disponible',
               'dias_inventario','nivel_urgencia']].head(15).to_string(index=False))

⚠️  Sin Cassandra — usando CSV para el análisis de stockout

🚨 ANÁLISIS DE STOCKOUT — 33 combinaciones en alerta crítica (<7 días):
                  sucursal      modelo  total_vendido  stock_disponible  dias_inventario nivel_urgencia
               CDMX_Centro 084 Campeón              7                 0                0      🔴 CRÍTICA
             CDMX_La_Villa      Urbano              3                 0                0      🔴 CRÍTICA
CDMX_Metrobus_Chilpancingo      Urbano              1                 0                0      🔴 CRÍTICA
     CDMX_Metrobus_Durango 084 Campeón              2                 0                0      🔴 CRÍTICA
              CDMX_Mixcoac      Urbano              1                 0                0      🔴 CRÍTICA
         CDMX_Plaza_Ermita 084 Campeón              1                 0                0      🔴 CRÍTICA
              CDMX_Polanco 084 Campeón              1                 0                0      🔴 CRÍTICA
               CDMX_Regina 084 Campe

---

## 4️⃣ ⚡ Consultas Optimizadas — Demostrando el poder de las Claves Primarias

Cassandra no necesita índices adicionales cuando la consulta usa la clave primaria. Aquí vemos la diferencia entre una query que SÍ usa la PK y una que no.


In [11]:
if session is not None:
    print(f"{'='*72}")
    print("⚡ EFICIENCIA DE QUERIES POR TIPO:")
    print(f"{'='*72}\n")

    demos = [
        ("✅ Query ÓPTIMA (usa PK completa)",
         "SELECT * FROM ventas_por_sucursal WHERE sucursal = 'CDMX_Polanco' LIMIT 20"),
        ("✅ Query BUENA (usa partición)",
         "SELECT modelo, SUM(cantidad) FROM ventas_por_sucursal WHERE sucursal = 'CDMX_Insurgentes' GROUP BY modelo"),
        ("⚠️  Query con ALLOW FILTERING (sin PK)",
         "SELECT * FROM ventas_por_sucursal WHERE metodo_pago = 'efectivo' LIMIT 20 ALLOW FILTERING"),
    ]

    for descripcion, cql in demos:
        t0 = time.perf_counter()
        try:
            rows = session.execute(cql).all()
            elapsed_ms = (time.perf_counter() - t0) * 1000
            print(f"  {descripcion}")
            print(f"     Tiempo: {elapsed_ms:6.2f} ms  |  Filas: {len(rows)}")
        except Exception as e:
            print(f"  {descripcion}")
            print(f"     ⚠️  Error: {str(e)[:80]}")
        print()

    print("💡 ALLOW FILTERING hace un scan completo de la tabla — muy lento en producción.")
    print("   Siempre diseñar la PK para que las queries más frecuentes la usen.")
else:
    print("⚠️  Sin Cassandra — mostrando el concepto:")
    print("""
  ✅ Query ÓPTIMA: WHERE sucursal = 'CDMX_Polanco'
     → Cassandra va directo a la partición de CDMX_Polanco. Ultra rápido.

  ⚠️  Query lenta: WHERE metodo_pago = 'efectivo'
     → Cassandra tiene que leer TODAS las particiones (scan completo).
     → En producción con millones de filas: segundos vs milisegundos.
""")

⚠️  Sin Cassandra — mostrando el concepto:

  ✅ Query ÓPTIMA: WHERE sucursal = 'CDMX_Polanco'
     → Cassandra va directo a la partición de CDMX_Polanco. Ultra rápido.

  ⚠️  Query lenta: WHERE metodo_pago = 'efectivo'
     → Cassandra tiene que leer TODAS las particiones (scan completo).
     → En producción con millones de filas: segundos vs milisegundos.



---

## 5️⃣ 📊 Dashboard Visual — Ventas, Inventario y Estacionalidad


In [12]:
# ── Cargar datos para gráficas ───────────────────────────────────────────
df_v = pd.read_csv(DATOS / "ventas.csv")
df_i = pd.read_csv(DATOS / "inventario.csv")
df_v['fecha']  = pd.to_datetime(df_v['fecha'])
df_i['fecha']  = pd.to_datetime(df_i['fecha'])
df_v['mes']    = df_v['fecha'].dt.month
df_v['año']    = df_v['fecha'].dt.year
df_v['año_mes']= df_v['fecha'].dt.to_period('M').astype(str)
df_v['estado'] = df_v['sucursal'].str.split('_').str[0]

print(f"📊 Datos para gráficas:")
print(f"   Ventas:     {len(df_v):,} registros | {df_v['año'].min()}-{df_v['año'].max()}")
print(f"   Inventario: {len(df_i):,} movimientos")

📊 Datos para gráficas:
   Ventas:     5,000 registros | 2021-2026
   Inventario: 2,000 movimientos


In [13]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('💎 Dashboard de Ventas e Inventario — Panam NoSQL',
             fontsize=18, fontweight='bold', color=AZUL, y=1.01)
fig.patch.set_facecolor(LIGHT_BG)

NOMBRES_MES = ['Ene','Feb','Mar','Abr','May','Jun',
               'Jul','Ago','Sep','Oct','Nov','Dic']

# ─── Gráfica 1: Top 10 modelos por unidades ──────────────────────────────
ax1 = axes[0, 0]
top10 = df_v.groupby('modelo')['cantidad'].sum().sort_values(ascending=True).tail(10)
colors_top = [ROJO if '084' in m else AZUL for m in top10.index]
bars = ax1.barh(top10.index, top10.values, color=colors_top, edgecolor='white', height=0.65)
for bar in bars:
    ax1.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
             f'{int(bar.get_width()):,}', va='center', fontsize=9, fontweight='bold')
ax1.set_xlabel('Unidades vendidas')
ax1.set_title('Top 10 Modelos\npor Unidades Vendidas')
ax1.legend(handles=[
    mpatches.Patch(color=ROJO, label='Familia 084'),
    mpatches.Patch(color=AZUL, label='Otros modelos'),
], fontsize=9, frameon=False, loc='lower right')

# ─── Gráfica 2: Distribución de tallas ────────────────────────────────────
ax2 = axes[0, 1]
tallas = df_v.groupby('talla')['cantidad'].sum()
colores_talla = [NARANJA if t <= 21 else AZUL for t in tallas.index]
bars2 = ax2.bar(tallas.index, tallas.values, color=colores_talla, edgecolor='white', width=0.75)
ax2.axvline(21.5, color='black', linestyle='--', lw=1, alpha=0.5, label='Infantil | Adulto')
ax2.set_xlabel('Talla mexicana')
ax2.set_ylabel('Unidades vendidas')
ax2.set_title('Demanda por Talla\n(Mexicana 12-30)')
ax2.legend(handles=[
    mpatches.Patch(color=NARANJA, label='Infantil (12-21)'),
    mpatches.Patch(color=AZUL, label='Adulto (22-30)'),
], fontsize=9, frameon=False)

# ─── Gráfica 3: Estacionalidad mensual ─────────────────────────────────────
ax3 = axes[0, 2]
est = df_v.groupby('mes')['cantidad'].sum()
est_colors = ['#C62828' if m in (7,8,12) else '#1565C0' for m in est.index]
bars3 = ax3.bar(est.index, est.values, color=est_colors, edgecolor='white', width=0.75)
for bar in bars3:
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+15,
             f'{int(bar.get_height()):,}', ha='center', fontsize=8, fontweight='bold')
ax3.set_xticks(range(1,13))
ax3.set_xticklabels(NOMBRES_MES, fontsize=9)
ax3.set_ylabel('Unidades vendidas')
ax3.set_title('Estacionalidad de Ventas\n(picos: Regreso a clases y Navidad)')
ax3.legend(handles=[
    mpatches.Patch(color='#C62828', label='Picos: Jul, Ago, Dic'),
    mpatches.Patch(color='#1565C0', label='Temporada normal'),
], fontsize=9, frameon=False)

# ─── Gráfica 4: Tendencia anual de ventas ────────────────────────────────
ax4 = axes[1, 0]
tend = df_v.groupby('año_mes')['cantidad'].sum()
ax4.fill_between(range(len(tend)), tend.values, alpha=0.25, color=AZUL)
ax4.plot(range(len(tend)), tend.values, color=AZUL, lw=2.5, marker='o', markersize=3)
# Promedio móvil
mm = pd.Series(tend.values).rolling(3, min_periods=1).mean()
ax4.plot(range(len(mm)), mm.values, color=ROJO, lw=2, linestyle='--', label='Media móvil 3m')
n = len(tend)
step = max(n//8, 1)
ax4.set_xticks(range(0, n, step))
ax4.set_xticklabels([str(tend.index[i]) for i in range(0, n, step)], rotation=35, ha='right', fontsize=8)
ax4.set_ylabel('Unidades vendidas')
ax4.set_title('Tendencia de Ventas\n(últimos 5 años)')
ax4.legend(frameon=False, fontsize=9)

# ─── Gráfica 5: Método de pago ────────────────────────────────────────────
ax5 = axes[1, 1]
pagos = df_v.groupby('metodo_pago')['cantidad'].sum().sort_values(ascending=False)
colors_pago = PALETTE[:len(pagos)]
wedges, texts, autotexts = ax5.pie(
    pagos.values, labels=None, colors=colors_pago, autopct='%1.1f%%',
    startangle=140, pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
    textprops={'fontsize': 11, 'fontweight': 'bold'},
)
for at in autotexts: at.set_color('white')
ax5.legend(handles=[mpatches.Patch(color=c, label=l)
                     for c, l in zip(colors_pago, pagos.index)],
           loc='lower center', ncol=2, bbox_to_anchor=(0.5,-0.22), frameon=False, fontsize=9)
ax5.set_title('Métodos de Pago\n(participación)')

# ─── Gráfica 6: Ventas por estado ────────────────────────────────────────
ax6 = axes[1, 2]
estado_ventas = df_v.groupby('estado')['cantidad'].sum().sort_values(ascending=True)
colors_estado = [ROJO if 'CDMX' in e else AZUL for e in estado_ventas.index]
bars6 = ax6.barh(estado_ventas.index, estado_ventas.values, color=colors_estado,
                  edgecolor='white', height=0.65)
for bar in bars6:
    ax6.text(bar.get_width()+5, bar.get_y()+bar.get_height()/2,
             f'{int(bar.get_width()):,}', va='center', fontsize=9, fontweight='bold')
ax6.set_xlabel('Unidades vendidas')
ax6.set_title('Ventas por Estado\n(participación regional)')
ax6.legend(handles=[
    mpatches.Patch(color=ROJO, label='CDMX'),
    mpatches.Patch(color=AZUL, label='Estados'),
], fontsize=9, frameon=False, loc='lower right')

plt.tight_layout(pad=2.5)
out_path = DATOS / "graficas_cassandra.png"
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n💾 Gráfica guardada en: {out_path}")


💾 Gráfica guardada en: c:\Users\PC\Desktop\Antigravity\Panam_BDN\data\graficas_cassandra.png


### 5.1 — Heatmap de stockout: Sucursal × Nivel de urgencia

In [18]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor(LIGHT_BG)

# ─── Gráfica 7: Heatmap de stockout ──────────────────────────────────
ax7 = axes[0]

COLORES_URG = {
    'OK':      VERDE,
    'MEDIA':   AMARILLO,
    'ALTA':    NARANJA,
    'CRÍTICA': ROJO,
}

# Crear pivot table: filas=modelos, columnas=estados, valores=conteo por urgencia
df_analisis['urg_display'] = df_analisis['nivel_urgencia'].map({
    '🔴 CRÍTICA': 'CRÍTICA',
    '🟠 ALTA':    'ALTA',
    '🟡 MEDIA':   'MEDIA',
    '🟢 OK':      'OK',
}).fillna('OK')

# Contar cuántas combinaciones hay en cada urgencia por estado y modelo
pivot_count = df_analisis.pivot_table(
    index='modelo', 
    columns='estado', 
    values='urg_display', 
    aggfunc=lambda x: x.value_counts().index[0] if len(x) > 0 else 'OK',  # Moda (urgencia más frecuente)
    fill_value='OK'
)

# Mapear a números para el heatmap
urgencia_map = {'OK': 0, 'MEDIA': 1, 'ALTA': 2, 'CRÍTICA': 3}
pivot_num = pivot_count.map(lambda x: urgencia_map.get(x, 0))

# Crear heatmap
cmap_urg = sns.color_palette([VERDE, AMARILLO, NARANJA, ROJO], as_cmap=True)
sns.heatmap(
    pivot_num, 
    ax=ax7, 
    cmap=cmap_urg, 
    vmin=0, 
    vmax=3,
    linewidths=0.5, 
    linecolor='white',
    cbar_kws={'label':'Urgencia (0=OK, 3=CRÍTICA)', 'shrink':0.8},
    annot=pivot_count.values,  # Mostrar el texto de urgencia
    fmt='',
    annot_kws={'size':10, 'weight':'bold'},
)

ax7.set_title('🚨 Distribución de Urgencia de Stockout por Modelo y Estado\n(color = urgencia máxima encontrada)',
              fontsize=13, fontweight='bold', pad=12, color=ROJO)
ax7.set_xlabel('Estado', fontsize=11)
ax7.set_ylabel('Modelo', fontsize=11)
ax7.tick_params(axis='x', rotation=35, labelsize=9)
ax7.tick_params(axis='y', rotation=0, labelsize=9)

# ─── Gráfica 8: Distribución de urgencia ─────────────────────────────
ax8 = axes[1]

conteo_urg = df_analisis['urg_display'].value_counts().reindex(
    ['CRÍTICA','ALTA','MEDIA','OK'], fill_value=0)
colores_urg_bar = [ROJO, NARANJA, AMARILLO, VERDE]

bars8 = ax8.bar(conteo_urg.index, conteo_urg.values,
                color=colores_urg_bar, edgecolor='white', linewidth=2, width=0.6)
for bar in bars8:
    h = int(bar.get_height())
    ax8.text(bar.get_x()+bar.get_width()/2, h+0.5,
             f'{h:,}', ha='center', va='bottom', fontsize=14, fontweight='bold')
ax8.set_xlabel('Nivel de urgencia', fontsize=12)
ax8.set_ylabel('Combinaciones sucursal-modelo', fontsize=12)
ax8.set_title('Distribución de\nNiveles de Urgencia de Stockout',
              fontsize=13, fontweight='bold', pad=12)

for bar, color in zip(bars8, colores_urg_bar):
    bar.set_edgecolor(color)
    bar.set_linewidth(2)

criticos = int(conteo_urg.get('CRÍTICA', 0))
if criticos > 0:
    msg = f'⚠️  {criticos} combinaciones en estado CRÍTICO'
else:
    msg = f'✅ {len(df_analisis)} combinaciones monitoreadas\n   Inventario en niveles aceptables'

ax8.text(0.5, 0.93, msg,
         transform=ax8.transAxes, ha='center', fontsize=11,
         color=ROJO if criticos > 0 else VERDE, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.4', 
                  facecolor='#FFCDD2' if criticos > 0 else '#C8E6C9', 
                  alpha=0.8))

plt.tight_layout(pad=3)
out_path2 = DATOS / "stockout_heatmap.png"
plt.savefig(out_path2, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n💾 Heatmap guardado en: {out_path2}")

print(f"\n{'='*72}")
print("📊 RESUMEN EJECUTIVO DE STOCKOUT:")
print(f"{'='*72}")
for nivel in ['CRÍTICA','ALTA','MEDIA','OK']:
    n = int(conteo_urg.get(nivel,0))
    icono = {'CRÍTICA':'🔴','ALTA':'🟠','MEDIA':'🟡','OK':'🟢'}[nivel]
    pct = (n / len(df_analisis) * 100) if len(df_analisis) > 0 else 0
    print(f"  {icono} {nivel:8s}: {n:>3} combinaciones ({pct:5.1f}%)")


💾 Heatmap guardado en: c:\Users\PC\Desktop\Antigravity\Panam_BDN\data\stockout_heatmap.png

📊 RESUMEN EJECUTIVO DE STOCKOUT:
  🔴 CRÍTICA :  33 combinaciones ( 15.1%)
  🟠 ALTA    :   0 combinaciones (  0.0%)
  🟡 MEDIA   :   0 combinaciones (  0.0%)
  🟢 OK      : 186 combinaciones ( 84.9%)


---

## ✅ Resumen del Notebook 04

### Lo que demostramos en vivo

| # | Proceso | Resultado |
|---|---------|-----------|
| 1️⃣ | Transformación de tipos CSV → Cassandra | `date`, `time`, `Decimal` correctos |
| 2️⃣ | Carga batch de 5,000 ventas | ~1,000 registros/s con `BatchStatement` |
| 3️⃣ | Carga batch de 2,000 movimientos de inventario | Datos distribuidos por estado |
| 4️⃣ | Análisis de stockout end-to-end | Alertas CRÍTICA/ALTA/MEDIA/OK |
| 5️⃣ | Range queries optimizadas (con y sin PK) | Diferencia de rendimiento medida |
| 6️⃣ | Dashboard de 8 gráficas | Insights de ventas, tallas y estacionalidad |

### Insights clave para la presentación de Panam

- **Talla más vendida**: verificar en la gráfica 2 cuál domina (esperado: 25)
- **Mes pico de ventas**: julio-agosto (regreso a clases) y diciembre (navidad)
- **Estado con más ventas**: CDMX concentra el mayor volumen (23 sucursales)
- **Modelo más popular**: la familia 084 lidera consistentemente
- **Stockouts críticos**: ver el heatmap para priorizar reabastecimientos

### Por qué Cassandra y no SQL aquí

- Range queries por fecha en Cassandra: **milisegundos** → en SQL sin índice: **segundos**
- Inserción batch de 5,000 filas: **2 segundos** → en SQL row-by-row: **30+ segundos**
- No hay JOINs posibles en Cassandra → obliga a un **modelo desnormalizado optimizado por consulta**
- Escalabilidad horizontal: agregar 10 sucursales más = agregar nodos, sin migración de esquema

➡️ Siguiente: construir el dashboard interactivo final para la presentación
